In [ ]:
!python gym_pybullet_drones/examples/train_navigate.py --timesteps 4000000

# Resume train the model

In [ ]:
!python gym_pybullet_drones/examples/train_navigate.py --resume_model results/navigate/stage_1_icarus/best_model.zip \
  --difficulty medium \
  --lr 1e-4 \
  --timesteps 300000 \
  --n_envs 4


In [ ]:
!python gym_pybullet_drones/examples/train_navigate.py --resume_model results/navigate/stage_3_icarus/best_model.zip \
  --difficulty hard \
  --lr 1e-4 \
  --timesteps 1000000 \
  --n_envs 4

# Evaluate

In [ ]:
!python gym_pybullet_drones/examples/test_navigate.py --difficulty hard --model_path results/navigate/stage_3_icarus/best_model.zip --record_video False --episodes 15

In [ ]:
!python gym_pybullet_drones/examples/test_navigate.py \
  --model_path results/navigate/stage_3_icarus/best_model.zip \
  --difficulty hard \
  --episodes 30 \
  --gui False \
  --show_camera False

# Deploy to tello drone

In [ ]:
!python gym_pybullet_drones/examples/deploy_tello.py \
  --model results/navigate/stage_3_icarus/best_model.zip \
  --target 0.5 0.0 0.5
  

In [ ]:
tello.end()

In [ ]:

from djitellopy import Tello
tello = Tello()
tello.connect()
print('Battery    :', tello.get_battery(), '%')
print('Temperature:', tello.get_highest_temperature(), 'C')
print('Firmware   :', tello.query_sdk_version())


In [ ]:

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import time, cv2, torch, numpy as np
from djitellopy import Tello

print('[1/4] Connecting to Tello...')
tello = Tello()
tello.connect()
tello.streamon()

print('[2/4] Initializing MiDaS Depth Estimator on Apple Silicon...')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
midas = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small').to(device).eval()
transforms = torch.hub.load('intel-isl/MiDaS', 'transforms').small_transform

print('[3/4] Warming up camera stream...')
frame_read = tello.get_frame_read()
time.sleep(1.5)

print('[4/4] Rendering Live HUD. Press [q] or [SPACE] in window to close.')
while True:
    frame = frame_read.frame
    if frame is None:
        continue
    
    # Run depth inference
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    input_tensor = transforms(rgb).to(device)
    with torch.no_grad():
        depth = midas(input_tensor)
        depth = torch.nn.functional.interpolate(
            depth.unsqueeze(1), size=(480, 640), mode='bicubic', align_corners=False
        ).squeeze().cpu().numpy()
    
    # Normalize and colormap
    d_min, d_max = np.percentile(depth, 2), np.percentile(depth, 98)
    depth_norm = np.clip((d_max - depth) / (d_max - d_min + 1e-6), 0.0, 1.0)
    depth_vis = cv2.applyColorMap((depth_norm * 255).astype(np.uint8), cv2.COLORMAP_TURBO)
    
    # Show side by side
    combined = np.hstack([cv2.resize(frame, (640, 480)), depth_vis])
    cv2.putText(combined, 'LIVE RGB FPV', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    cv2.putText(combined, 'NEURAL DEPTH (MiDaS)', (660, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
    
    cv2.imshow('Tello Talent Vision Test (Press Q to Exit)', combined)
    if cv2.waitKey(1) & 0xFF in [ord('q'), 32]:
        break

tello.streamoff()
cv2.destroyAllWindows()
print('[DONE] Vision pipeline verified successfully!')


In [ ]:
!python gym_pybullet_drones/examples/check_tello_vps.py
